# Senate PTR Scraper — Electronic Filings Only

This notebook focuses only on **electronic Senate Periodic Transaction Reports (PTRs)**.

### What it does
- searches the official Senate eFD PTR index
- identifies both electronic (`/ptr/`) and paper (`/paper/`) filings
- **scrapes only electronic PTRs**
- parses the official HTML transaction table
- saves the official HTML, filing index, transaction CSV, status CSV, and Excel workbook
- resumes from an existing electronic transaction CSV when available

### Paper filings
Paper filings are **not downloaded or parsed in this notebook**. They are still kept in the filing index and marked `paper_deferred` in the status file so they can be handled as a separate project later.

This intentionally removes the paper downloader, PDF/image rendering, Qwen vision model, paper review tables, and combined paper/electronic output from the older V2 notebook.


In [ ]:
# CELL 1 — Install only the packages this electronic scraper needs

!pip -q install beautifulsoup4 lxml openpyxl

print("Packages installed.")


In [ ]:
# CELL 2 — Imports

import re
import time
import json
import hashlib
from pathlib import Path
from datetime import datetime, date
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup

from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 140)

print("Imports ready.")


In [ ]:
# CELL 3 — Mount Google Drive and create folders

from google.colab import drive
drive.mount("/content/drive")

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Congressional Trading Data/Senate_PTRs"
)

RAW_DIR = PROJECT_ROOT / "01 Official Senate PTR HTML"
INDEX_DIR = PROJECT_ROOT / "02 Senate PTR Filing Indexes"
DATA_DIR = PROJECT_ROOT / "03 Parsed Senate PTR Transaction Data"
STATUS_DIR = PROJECT_ROOT / "04 Scrape Checkpoints and Status"

for folder in [
    PROJECT_ROOT,
    RAW_DIR,
    INDEX_DIR,
    DATA_DIR,
    STATUS_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)


In [ ]:
# CELL 4 — Configuration

START_DATE = "2021-01-01"
END_DATE = date.today().isoformat()

BATCH_SIZE = 100
REQUEST_DELAY = 0.40

# Save the official HTML page for each electronic PTR.
SAVE_ELECTRONIC_HTML = True

# Reuse already-parsed electronic reports when a prior CSV exists.
RESUME_EXISTING = True

print("Filing range:", START_DATE, "through", END_DATE)


In [ ]:
# CELL 5 — Establish Senate eFD session

ROOT = "https://efdsearch.senate.gov"
LANDING_URL = f"{ROOT}/search/home/"
SEARCH_URL = f"{ROOT}/search/"
REPORTS_URL = f"{ROOT}/search/report/data/"


def build_session():
    client = requests.Session()

    retry = Retry(
        total=3,
        connect=3,
        read=3,
        status=3,
        backoff_factor=1.0,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET", "POST"],
        raise_on_status=False,
    )

    adapter = HTTPAdapter(max_retries=retry)
    client.mount("https://", adapter)

    client.headers.update({
        "User-Agent": (
            "Mozilla/5.0 (X11; Linux x86_64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/124.0 Safari/537.36"
        ),
        "Accept-Language": "en-US,en;q=0.9",
    })

    return client


def get_page_csrf(html):
    soup = BeautifulSoup(html, "lxml")
    token = soup.find(attrs={"name": "csrfmiddlewaretoken"})
    return token.get("value") if token else None


def accept_senate_agreement(client):
    r = client.get(LANDING_URL, timeout=30)

    if r.status_code == 403:
        raise RuntimeError(
            "Senate eFD returned HTTP 403 before the scrape began. "
            "This usually means the current Colab IP is being blocked. "
            "Try another runtime later or use a local runtime."
        )

    r.raise_for_status()

    form_csrf = get_page_csrf(r.text)
    if not form_csrf:
        raise RuntimeError("Could not locate the Senate agreement CSRF token.")

    accepted = client.post(
        LANDING_URL,
        data={
            "csrfmiddlewaretoken": form_csrf,
            "prohibition_agreement": "1",
        },
        headers={"Referer": LANDING_URL},
        timeout=30,
        allow_redirects=True,
    )

    if accepted.status_code == 403:
        raise RuntimeError(
            "Senate eFD returned HTTP 403 while accepting the agreement."
        )

    accepted.raise_for_status()

    csrf = client.cookies.get("csrftoken") or client.cookies.get("csrf")

    if not csrf:
        raise RuntimeError(
            "The Senate agreement was accepted, but no CSRF cookie was returned."
        )

    return csrf


session = build_session()
csrf_token = accept_senate_agreement(session)

print("Senate eFD session established successfully.")
print("CSRF token available:", bool(csrf_token))


In [ ]:
# CELL 6 — Search the official Senate PTR filing index

def mmddyyyy(iso_date):
    return pd.to_datetime(iso_date).strftime("%m/%d/%Y")


def search_ptr_batch(client, csrf, start=0, length=100):
    payload = {
        "start": str(start),
        "length": str(length),
        "report_types": "[11]",
        "filer_types": "[1]",
        "submitted_start_date": f"{mmddyyyy(START_DATE)} 00:00:00",
        "submitted_end_date": f"{mmddyyyy(END_DATE)} 23:59:59",
        "csrfmiddlewaretoken": csrf,
    }

    r = client.post(
        REPORTS_URL,
        data=payload,
        headers={
            "Referer": SEARCH_URL,
            "X-CSRFToken": csrf,
            "X-Requested-With": "XMLHttpRequest",
        },
        timeout=60,
    )

    if r.status_code == 403:
        raise RuntimeError("Senate report search returned HTTP 403.")

    r.raise_for_status()

    try:
        return r.json()
    except Exception:
        preview = r.text[:500].replace("\n", " ")
        raise RuntimeError(
            "Senate search did not return JSON. "
            f"Response started with: {preview}"
        )


def parse_search_result(row):
    raw = list(row)

    first_name = str(raw[0]).strip() if len(raw) > 0 else ""
    last_name = str(raw[1]).strip() if len(raw) > 1 else ""
    office = str(raw[2]).strip() if len(raw) > 2 else ""
    link_html = str(raw[3]) if len(raw) > 3 else ""
    filing_date = str(raw[4]).strip() if len(raw) > 4 else ""

    soup = BeautifulSoup(link_html, "lxml")
    anchor = soup.find("a")

    relative_url = anchor.get("href") if anchor else ""
    report_url = urljoin(ROOT, relative_url) if relative_url else ""
    report_title = anchor.get_text(" ", strip=True) if anchor else ""

    # Keep paper filings in the index, but do not process them here.
    match = re.search(
        r"/search/view/(ptr|paper)/([0-9A-Za-z-]+)/?",
        report_url,
        flags=re.I,
    )

    report_kind = match.group(1).lower() if match else ""
    report_id = match.group(2) if match else ""

    if report_kind == "ptr":
        filing_format = "electronic_html"
    elif report_kind == "paper":
        filing_format = "paper_scan"
    else:
        filing_format = "unknown"

    report_key = report_id or hashlib.sha256(
        report_url.encode("utf-8")
    ).hexdigest()

    return {
        "first_name": first_name,
        "last_name": last_name,
        "filer_name": " ".join(x for x in [first_name, last_name] if x),
        "office": office,
        "filing_date": filing_date,
        "report_title": report_title,
        "report_id": report_id,
        "report_key": report_key,
        "filing_format": filing_format,
        "report_url": report_url,
    }


filings = []
offset = 0
page = 0
MAX_SEARCH_PAGES = 500  # hard safety cap; ~50,000 filings at BATCH_SIZE=100

while True:
    page += 1

    if page > MAX_SEARCH_PAGES:
        raise RuntimeError(
            f"Exceeded MAX_SEARCH_PAGES ({MAX_SEARCH_PAGES}) while paginating "
            "the Senate PTR search. Aborting to avoid an unbounded loop -- "
            "check the API response shape, or raise this cap deliberately."
        )

    result = search_ptr_batch(
        session,
        csrf_token,
        start=offset,
        length=BATCH_SIZE,
    )

    rows = result.get("data", [])
    if not rows:
        break

    filings.extend(parse_search_result(row) for row in rows)

    filtered_total = result.get("recordsFiltered")
    offset += len(rows)

    if filtered_total is not None:
        print(f"Found {len(filings)} / {filtered_total} filings")
    else:
        print(f"Found {len(filings)} filings")

    if len(rows) < BATCH_SIZE:
        break

    if filtered_total is not None and offset >= int(filtered_total):
        break

    time.sleep(REQUEST_DELAY)


filings_df = pd.DataFrame(filings)

if not filings_df.empty:
    filings_df["filing_date"] = (
        pd.to_datetime(filings_df["filing_date"], errors="coerce")
        .dt.strftime("%Y-%m-%d")
    )

    filings_df = (
        filings_df
        .drop_duplicates(subset=["report_key"], keep="first")
        .reset_index(drop=True)
    )

FILING_INDEX_CSV = INDEX_DIR / "Senate_PTR_Filing_Index.csv"
filings_df.to_csv(FILING_INDEX_CSV, index=False)

print()
display(
    filings_df["filing_format"]
    .value_counts(dropna=False)
    .rename_axis("filing_format")
    .reset_index(name="reports")
)

print("Total filing rows:", len(filings_df))
print("Saved:", FILING_INDEX_CSV)


In [ ]:
# CELL 7 — Electronic PTR parsing helpers

EXPECTED_HEADERS = [
    "#",
    "Transaction Date",
    "Owner",
    "Ticker",
    "Asset Name",
    "Asset Type",
    "Type",
    "Amount",
    "Comment",
]


def clean_text(value):
    if value is None:
        return None

    text = re.sub(r"\s+", " ", str(value)).strip()

    if text in {"", "--", "—", "–", "None", "null"}:
        return None

    return text


def normalize_header(value):
    text = clean_text(value) or ""
    text = text.replace("Transac- tion", "Transaction")
    text = text.replace("TransactionDate", "Transaction Date")
    return re.sub(r"\s+", " ", text).strip()


def parse_money_range(amount_text):
    text = clean_text(amount_text)
    if not text:
        return None, None

    nums = [
        int(x.replace(",", ""))
        for x in re.findall(r"\$\s*([0-9][0-9,]*)", text)
    ]

    if not nums:
        return None, None

    lower = text.lower()

    if "over" in lower or "more than" in lower:
        return nums[0], None

    if len(nums) >= 2:
        return nums[0], nums[1]

    return nums[0], nums[0]


def iso_date_or_original(value):
    value = clean_text(value)
    if not value:
        return None

    parsed = pd.to_datetime(value, errors="coerce")

    if pd.isna(parsed):
        return value

    return parsed.strftime("%Y-%m-%d")


def now_utc():
    return datetime.utcnow().replace(microsecond=0).isoformat() + "Z"


def find_transaction_table(soup):
    for table in soup.find_all("table"):
        headers = [
            normalize_header(th.get_text(" ", strip=True))
            for th in table.find_all("th")
        ]

        joined = " | ".join(headers).lower()

        if (
            "transaction" in joined
            and "owner" in joined
            and "ticker" in joined
            and "amount" in joined
        ):
            return table

    for table in soup.find_all("table"):
        tbody = table.find("tbody")
        if tbody and tbody.find("tr"):
            if len(tbody.find("tr").find_all("td")) >= 8:
                return table

    return None


def extract_headers(table):
    first_row = table.find("tr")

    if first_row:
        headers = [
            normalize_header(x.get_text(" ", strip=True))
            for x in first_row.find_all(["th", "td"])
        ]

        if len(headers) >= 7:
            return headers

    return EXPECTED_HEADERS


def canonicalize_row(row_dict):
    lookup = {}

    for key, value in row_dict.items():
        norm = re.sub(r"[^a-z0-9]+", "", str(key).lower())
        lookup[norm] = clean_text(value)

    return {
        "item_number": lookup.get("") or lookup.get("number") or lookup.get("item"),
        "transaction_date": lookup.get("transactiondate") or lookup.get("date"),
        "owner": lookup.get("owner"),
        "ticker": lookup.get("ticker"),
        "asset_name": lookup.get("assetname"),
        "asset_type": lookup.get("assettype"),
        "transaction_type": lookup.get("type") or lookup.get("transactiontype"),
        "amount": lookup.get("amount"),
        "comment": lookup.get("comment"),
    }


def parse_electronic_ptr(html, filing):
    soup = BeautifulSoup(html, "lxml")
    table = find_transaction_table(soup)

    if table is None:
        return [], "No electronic transaction table found"

    headers = extract_headers(table)
    body_rows = (
        table.find("tbody").find_all("tr")
        if table.find("tbody")
        else table.find_all("tr")[1:]
    )

    output = []

    for row_position, tr in enumerate(body_rows, start=1):
        values = [
            clean_text(td.get_text(" ", strip=True))
            for td in tr.find_all("td")
        ]

        if not values:
            continue

        use_headers = headers

        if len(values) == 9 and len(headers) != 9:
            use_headers = EXPECTED_HEADERS

        if len(use_headers) < len(values):
            use_headers = use_headers + [
                f"extra_{n}"
                for n in range(len(use_headers), len(values))
            ]

        tx = canonicalize_row(
            dict(zip(use_headers[:len(values)], values))
        )

        transaction_date = iso_date_or_original(tx["transaction_date"])
        amount_min, amount_max = parse_money_range(tx["amount"])
        item_number = tx.get("item_number") or str(row_position)

        stable = "|".join([
            filing.get("report_id") or filing.get("report_key") or "",
            str(item_number),
            transaction_date or "",
            tx.get("owner") or "",
            tx.get("ticker") or "",
            tx.get("asset_name") or "",
            tx.get("transaction_type") or "",
            tx.get("amount") or "",
            tx.get("comment") or "",
        ])

        transaction_id = hashlib.sha256(
            stable.encode("utf-8")
        ).hexdigest()

        output.append({
            "transaction_id": transaction_id,
            "chamber": "Senate",
            "first_name": filing.get("first_name"),
            "last_name": filing.get("last_name"),
            "filer_name": filing.get("filer_name"),
            "office": filing.get("office"),
            "filing_date": filing.get("filing_date"),
            "report_title": filing.get("report_title"),
            "report_id": filing.get("report_id"),
            "report_key": filing.get("report_key"),
            "report_url": filing.get("report_url"),
            "item_number": item_number,
            "transaction_date": transaction_date,
            "owner": tx.get("owner"),
            "ticker": tx.get("ticker"),
            "asset_name": tx.get("asset_name"),
            "asset_type": tx.get("asset_type"),
            "transaction_type": tx.get("transaction_type"),
            "amount": tx.get("amount"),
            "amount_min": amount_min,
            "amount_max": amount_max,
            "comment": tx.get("comment"),
            "source_format": "electronic_html",
            "extraction_method": "official_html_table",
            "verification_status": "source_structured",
            "scraped_at_utc": now_utc(),
        })

    return output, None


print("Electronic parser ready.")


In [ ]:
# CELL 8 — Scrape only electronic PTRs

TRANSACTIONS_CSV = DATA_DIR / "Senate_PTR_Transactions_Electronic.csv"

# Backward-compatible resume sources from earlier notebook versions.
resume_candidates = [
    TRANSACTIONS_CSV,
    DATA_DIR / "Senate_PTR_Transactions_Electronic_V2.csv",
    DATA_DIR / "Senate_PTR_Transactions.csv",
]

existing_df = pd.DataFrame()
resume_source = None

if RESUME_EXISTING:
    for candidate in resume_candidates:
        if candidate.exists():
            existing_df = pd.read_csv(candidate, low_memory=False)
            resume_source = candidate
            break

if not existing_df.empty:
    # Keep only electronic rows if an older mixed file is ever supplied.
    if "source_format" in existing_df.columns:
        existing_df = existing_df[
            existing_df["source_format"].fillna("electronic_html")
            == "electronic_html"
        ].copy()

    # Remove old paper-only columns if they came from V2.
    existing_df = existing_df.drop(
        columns=[
            "paper_page_number",
            "paper_row_index",
            "paper_confidence",
        ],
        errors="ignore",
    )

    if "report_key" not in existing_df.columns:
        existing_df["report_key"] = existing_df["report_id"]

    existing_df["source_format"] = "electronic_html"
    existing_df["extraction_method"] = "official_html_table"
    existing_df["verification_status"] = "source_structured"

    print("Resuming from:", resume_source)
    print("Existing electronic transactions:", len(existing_df))


completed_report_ids = set(
    existing_df["report_id"].dropna().astype(str)
) if not existing_df.empty else set()

electronic_filings = filings_df[
    filings_df["filing_format"] == "electronic_html"
].copy()

to_scrape = electronic_filings[
    ~electronic_filings["report_id"].astype(str).isin(completed_report_ids)
].reset_index(drop=True)

print("Electronic PTR filings in index:", len(electronic_filings))
print("Electronic PTR filings left to scrape:", len(to_scrape))

new_rows = []
electronic_status = []

for i, filing_row in to_scrape.iterrows():
    filing = filing_row.to_dict()

    try:
        r = session.get(
            filing["report_url"],
            headers={"Referer": SEARCH_URL},
            timeout=60,
        )

        if r.status_code == 403:
            raise RuntimeError("HTTP 403")

        r.raise_for_status()

        if SAVE_ELECTRONIC_HTML:
            html_path = RAW_DIR / f"{filing['report_id']}.html"
            html_path.write_text(r.text, encoding="utf-8")

        rows, error = parse_electronic_ptr(r.text, filing)
        new_rows.extend(rows)

        electronic_status.append({
            "report_id": filing["report_id"],
            "status": "ok" if error is None else "needs_review",
            "transactions_found": len(rows),
            "error": error or "",
        })

        print(
            f"[{i+1}/{len(to_scrape)}] "
            f"{filing['filer_name']} | "
            f"{len(rows)} transactions"
        )

    except Exception as exc:
        electronic_status.append({
            "report_id": filing["report_id"],
            "status": "error",
            "transactions_found": 0,
            "error": repr(exc),
        })

        print(
            f"[{i+1}/{len(to_scrape)}] ERROR | "
            f"{filing['filer_name']} | {repr(exc)}"
        )

    time.sleep(REQUEST_DELAY)


frames = []

if not existing_df.empty:
    frames.append(existing_df)

if new_rows:
    frames.append(pd.DataFrame(new_rows))

if frames:
    transactions_df = pd.concat(
        frames,
        ignore_index=True,
        sort=False,
    )

    transactions_df = transactions_df.drop_duplicates(
        subset=["transaction_id"],
        keep="last",
    ).reset_index(drop=True)
else:
    transactions_df = pd.DataFrame()

transactions_df.to_csv(TRANSACTIONS_CSV, index=False)

print()
print("Electronic transactions available:", len(transactions_df))
print("Saved:", TRANSACTIONS_CSV)


In [ ]:
# CELL 9 — Build filing status table

new_status_map = {
    row["report_id"]: row
    for row in electronic_status
}

transaction_count_map = (
    transactions_df.groupby("report_id").size().to_dict()
    if not transactions_df.empty
    else {}
)

status_rows = []

for _, filing_row in filings_df.iterrows():
    filing = filing_row.to_dict()

    report_id = filing["report_id"]
    filing_format = filing["filing_format"]

    if filing_format == "paper_scan":
        status = "paper_deferred"
        transactions_found = 0
        error = "Paper filing intentionally deferred to a future project."

    elif filing_format == "electronic_html":
        current = new_status_map.get(report_id)

        if current:
            status = current["status"]
            transactions_found = current["transactions_found"]
            error = current["error"]
        elif report_id in transaction_count_map:
            status = "ok_reused"
            transactions_found = int(transaction_count_map[report_id])
            error = ""
        else:
            status = "electronic_missing"
            transactions_found = 0
            error = "Electronic filing found in index but no parsed transactions are available."

    else:
        status = "unknown_format"
        transactions_found = 0
        error = "Unrecognized Senate report URL format."

    status_rows.append({
        "report_id": report_id,
        "report_key": filing["report_key"],
        "filer_name": filing.get("filer_name"),
        "filing_date": filing.get("filing_date"),
        "filing_format": filing_format,
        "report_url": filing.get("report_url"),
        "status": status,
        "transactions_found": transactions_found,
        "error": error,
        "checked_at_utc": now_utc(),
    })


status_df = pd.DataFrame(status_rows)

assert len(status_df) == len(filings_df)
assert status_df["report_key"].nunique() == len(filings_df)

STATUS_CSV = STATUS_DIR / "Senate_PTR_Scrape_Status.csv"
status_df.to_csv(STATUS_CSV, index=False)

display(
    status_df["status"]
    .value_counts(dropna=False)
    .rename_axis("status")
    .reset_index(name="reports")
)

print("Saved:", STATUS_CSV)


In [ ]:
# CELL 10 — Build final Excel workbook

OUTPUT_XLSX = DATA_DIR / "Senate_PTR_Transactions_Electronic.xlsx"

with pd.ExcelWriter(
    OUTPUT_XLSX,
    engine="openpyxl",
) as writer:

    transactions_df.to_excel(
        writer,
        sheet_name="Transactions",
        index=False,
    )

    filings_df.to_excel(
        writer,
        sheet_name="Filings",
        index=False,
    )

    status_df.to_excel(
        writer,
        sheet_name="Scrape Status",
        index=False,
    )


electronic_filings_count = int(
    (filings_df["filing_format"] == "electronic_html").sum()
)

paper_filings_count = int(
    (filings_df["filing_format"] == "paper_scan").sum()
)

print("SENATE PTR ELECTRONIC SCRAPE COMPLETE")
print("====================================")
print("All PTR filings indexed:", len(filings_df))
print("Electronic filings:", electronic_filings_count)
print("Paper filings deferred:", paper_filings_count)
print("Electronic transactions:", len(transactions_df))
print()
print("Saved:")
print(" ", TRANSACTIONS_CSV)
print(" ", OUTPUT_XLSX)
print(" ", FILING_INDEX_CSV)
print(" ", STATUS_CSV)


## Paper project note

Paper PTRs are intentionally outside this notebook.

They are still visible in:

- `Senate_PTR_Filing_Index.csv` with `filing_format = paper_scan`
- `Senate_PTR_Scrape_Status.csv` with `status = paper_deferred`

That preserves the list of paper filings without carrying the paper downloader, PDF/image processing, vision model, or review workflow in the electronic scraper.

When you return to paper filings later, make them a separate notebook that reads the filing index and processes only rows where `filing_format == "paper_scan"`.
